# Understanding Model Parameters, Quantization, and Prompt Behavior

This notebook explains:
- What model parameters are  
- What quantization is and why it matters  
- Why prompt behavior changes (especially with few-shot prompts)  
- How token limits affect model outputs  
- Practical experiments with FP32 vs INT8 inference  



In [3]:
!pip install transformers torch accelerate bitsandbytes

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
import torch

In [6]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model_fp32 = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    dtype=torch.float32
)

## What are Model Parameters?

Model parameters are the internal values learned during training.
They store:
- language understanding
- grammar
- relationships between words

More parameters = more expressive power.

Examples:
- Small model → fewer patterns
- Large model → richer understanding

## Tokenization and Context Length
LLMs process **tokens**, not words.

For `flan-t5-base`:
- Max input length = **512 tokens**
- Anything beyond this is silently truncated

In [8]:
def count_tokens(text):
    return len(tokenizer.tokenize(text))

In [9]:
sample_text = "The app crashes every time I upload a file."
print("Token count:", count_tokens(sample_text))

Token count: 11


#Why Prompts Fail (Token Overflow):
If the token count exceeds the model limit (512):
- Earlier context is ignored
- Few-shot learning stops working
- Output quality drops

This happens silently.

## What is Quantization?

Quantization reduces numerical precision to make models:
- Faster
- Smaller
- Easier to run on local machines

In [10]:
# Load INT8 model
quant_config = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto"
)

Numerical intuition (VERY IMPORTANT)
FP32 representation:
0.123456789

INT8 representation:
0.12


Now imagine millions of these approximations stacked together.

Result:

Less nuance

Slight loss in reasoning

Still great for summarization

#FP32 vs INT8 Comparison

In [11]:
prompt = "Summarize: The app crashes when uploading files."

# FP32
out_fp32 = model_fp32.generate(
    **tokenizer(prompt, return_tensors="pt"),
    max_new_tokens=30
)

# INT8
out_int8 = model_int8.generate(
    **tokenizer(prompt, return_tensors="pt"),
    max_new_tokens=30
)

print("FP32 Output:", tokenizer.decode(out_fp32[0], skip_special_tokens=True))
print("INT8 Output:", tokenizer.decode(out_int8[0], skip_special_tokens=True))

FP32 Output: It crashes when uploading files.
INT8 Output: Uninstall the app.


INT8 models:
- Use reduced precision
- Lose small numerical details
- Are more sensitive to bad prompts

This is why:
✔ Clear prompts matter  
✔ Short prompts work better  
✔ One-shot > many-shot


## Key Learnings

- Models think in **tokens**, not words
- Token overflow silently breaks prompts
- One-shot prompting often works best
- Quantization trades accuracy for speed
- Prompt clarity matters more than prompt length
- Models think in tokens, not words.
- More examples ≠ better understanding.
- Quantization = smaller, faster brain.
- One-shot prompting is often optimal.